# Frozen but Adaptive: cross-victim RL policy pilot

This notebook is a thin analysis surface for the tested `rl_transfer` package. The local synthetic run checks infrastructure only; it is not evidence for the paper claim. T1 means frozen score-based deployment, while T3 means limited target fine-tuning.

In [ ]:
import json
from collections import Counter
from pathlib import Path

from rl_transfer.cli import run_research_smoke
from rl_transfer.registry import VictimRegistry
from rl_transfer.research_metrics import AttackOutcome, asr_at_budgets, asr_query_auc


## Protocol and victim registry

In [ ]:
pilot_config = json.loads(Path('configs/rl_transfer/pilot.json').read_text())
victim_registry = VictimRegistry.from_json(Path('configs/victims/imagenet1k.json'))
print(pilot_config['primary_threat_model'])
print('families:', sorted({victim.family for victim in victim_registry.victims}))
print('registry digest:', victim_registry.digest())


## Run or load the deterministic CPU research smoke

In [ ]:
result_path = Path('output/rl_transfer/research_smoke.json')
if not result_path.exists():
    run_research_smoke(result_path, seed=7)
result = json.loads(result_path.read_text())
result['manifest']


## Frozen boundary: T1 versus T3

In [ ]:
frozen = result['frozen_t1']
print('T1 policy unchanged:', frozen['policy_digest_before'] == frozen['policy_digest_after'])
print('T1 total target calls:', frozen['total_target_calls'])
print('T3 is reported separately:', result['manifest']['t3_is_comparison_only'])


## ASR-query curve and normalized AUC

In [ ]:
outcomes = [AttackOutcome(frozen['clean_correct'], frozen['query_to_success'])]
curve = asr_at_budgets(outcomes, [0, 5, 10])
print('ASR by budget:', curve)
print('normalized AUC:', asr_query_auc(curve))


## Action-frequency shortcut diagnostic

In [ ]:
action_counts = Counter(frozen['actions'])
print('action histogram:', dict(sorted(action_counts.items())))
print('Compare this distribution with fixed and random baselines before claiming history use.')


## Week-7 go/no-go checklist

Proceed to the full family holdout only after the frozen recurrent policy beats matched fixed/random controls across several held-out models, population training improves over single-source training, and results are stable across seeds. The smoke result does not satisfy these gates.